# RFP 100건 RAG 데이터셋 구축 파이프라인 (Kaggle)

100건의 실제 RFP(제안요청서) 원문(HWP/HWPX/PDF)을 문서 로딩 -> 정제 -> 표/이미지 구조화 ->
청킹 -> 도메인 사전 -> 하이브리드 인덱싱 -> RAG 평가용 벤치마크까지 처리하는 파이프라인입니다.

**Kaggle Notebook 사용법**: Notebook Settings에서 Internet을 켜고, 원본 문서(zip 또는 hwp/hwpx/pdf 폴더)와
`data_list.csv`를 Add Input으로 연결한 뒤 위에서 아래로 순서대로 실행하세요. **Accelerator를 GPU(T4 등)로
켜두는 것을 권장합니다** - 6단계 Dense 임베딩(BGE-m3, 청크 수천 개)이 GPU에서 훨씬 빠릅니다(자동 감지 후
배치 크기도 늘려 씀). OCR은 GPU가 아닌 CPU 기반 Tesseract를 쓰므로 GPU를 켜도 켜지 않아도 OCR 속도는
동일합니다. hwp 추출도 이 노트북은 multiprocessing fork 대신 신호(signal) 기반 타임아웃만 쓰므로(과거
fork 기반 병렬 추출 + GPU OCR을 같은 런타임에서 섞었을 때 CUDA 컨텍스트가 이미 뜬 프로세스를 fork하며
데드락에 빠졌던 문제가 실측으로 확인된 바 있음), GPU를 켠 채로 실행해도 그 문제가 재발하지 않습니다.

## 알려진 함정과 이 노트북의 대응
- **HWP 파싱 라이브러리(`hwp-hwpx-parser` 1.0.0) 자체 버그**: 특정 바이트 경계에서 컨트롤 문자를 만나면
  인덱스가 전진하지 않아 무한 루프에 빠집니다(실측: hwp 96건 중 96건 전부 재현). 라이브러리 소스를 직접
  읽어 정확한 위치를 확인하고, 원본 로직과 동일한 전진 폭을 추가하는 몽키패치로 수정합니다(패치 전/후
  단위 테스트로 검증됨).
- **파일명 유니코드 정규화 불일치**: 원본 zip 안 실제 파일명은 NFD(자소분리)인 경우가 있고, `data_list.csv`의
  `파일명` 컬럼은 NFC입니다. 그대로 매칭하면 CSV 메타데이터를 못 찾습니다 -> NFC 정규화 후 매칭합니다.
- **문서 내 제어 아티팩트**: `\x02`, `\x03` 등 HWP 내부 제어 바이너리가 텍스트에 섞여 나올 수 있어,
  유니코드 카테고리 기반으로 원문/제거 로그를 모두 남기며 정제합니다.
- **표/이미지의 원문 내 위치 보존**: 텍스트를 통째로 이어붙이면 표/이미지가 어디 있었는지 사라집니다.
  `hwp-hwpx-parser`의 `ExtractOptions`가 표는 Markdown, 이미지는 `[IMAGE: 파일명]` 마커로 원래 위치에
  인라인으로 끼워 넣어주는 기능을 그대로 활용해 순서가 있는 요소(`elements`) 리스트로 복원합니다.
- **문서 메타데이터는 재추측하지 않음**: `data_list.csv`에 이미 발주기관/사업명/사업금액/공고일자 등
  정확한 메타데이터가 있으므로, 파일명(NFC 정규화)으로 조인해서 그대로 사용합니다. 본문에서 정규식으로
  다시 추측하지 않습니다.

## 산출물 (`OUTPUT_ROOT` 아래)
`documents.jsonl`(canonical 문서 단위 메타데이터), `elements.jsonl`(순서 보존 문단/표/이미지 요소),
`tables.jsonl`/`table_cells.jsonl`, `images.jsonl`, `artifact_logs.jsonl`(정제 중 제거된 내용 감사 로그),
`chunk_dataset.jsonl`, `domain_dictionary.jsonl`, `rag_benchmark.jsonl`, `errors.jsonl`,
`bm25_index.pkl`/`dense_index.faiss`/`dense_embeddings.npy`, `quality_report.json`, `README.md`,
그리고 전체를 묶은 zip.

In [ ]:
# Kaggle 의존성 설치 (Notebook Settings에서 Internet을 켜고 최초 1회 실행)
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-kor tesseract-ocr-eng > /dev/null
!pip -q install hwp-hwpx-parser pymupdf pytesseract pillow pandas numpy tqdm pyarrow \
    kiwipiepy rank-bm25 sentence-transformers faiss-cpu scikit-learn transformers

In [ ]:
# 입력 탐색: Kaggle Add Input으로 연결한 zip 또는 폴더에서 hwp/hwpx/pdf/csv를 찾아
# 작업 디렉터리로 모읍니다.
from pathlib import Path
import shutil
import zipfile

KAGGLE_INPUT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/rfp_work')
INPUT_ROOT = WORK_ROOT / 'input'
OUTPUT_ROOT = Path('/kaggle/working/rfp_rag_dataset')
IMAGE_DIR = OUTPUT_ROOT / 'images'
for p in (INPUT_ROOT, OUTPUT_ROOT, IMAGE_DIR):
    p.mkdir(parents=True, exist_ok=True)

zip_files = sorted(KAGGLE_INPUT.rglob('*.zip'))
print(f'ZIP 파일 {len(zip_files)}개 발견:', [z.name for z in zip_files])
for zip_path in zip_files:
    extract_dir = INPUT_ROOT / zip_path.stem
    if not extract_dir.exists():
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(extract_dir)

SUPPORTED_DOC = {'.hwp', '.hwpx', '.pdf'}
for src in KAGGLE_INPUT.rglob('*'):
    if src.is_file() and (src.suffix.lower() in SUPPORTED_DOC or src.suffix.lower() == '.csv'):
        rel = src.relative_to(KAGGLE_INPUT)
        dst = INPUT_ROOT / rel
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)

source_files = sorted(p for p in INPUT_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in SUPPORTED_DOC)
ext_counts = {}
for p in source_files:
    ext_counts[p.suffix.lower()] = ext_counts.get(p.suffix.lower(), 0) + 1
print(f'발견 문서: {len(source_files)}건 {ext_counts}')
assert source_files, 'HWP/HWPX/PDF 파일을 찾지 못했습니다. Add Input 설정을 확인하세요.'

csv_candidates = sorted(INPUT_ROOT.rglob('data_list.csv')) or sorted(INPUT_ROOT.rglob('*.csv'))
print(f'CSV 후보: {[c.name for c in csv_candidates]}')

In [ ]:
# 공통 유틸: 정규화, 정제, ID 생성
import hashlib
import re
import unicodedata as _ud
from datetime import datetime, timezone

SCHEMA_VERSION = '1.0.0'
PARSED_AT = datetime.now(timezone.utc).isoformat()


def nfc(s):
    return _ud.normalize('NFC', str(s)) if s is not None else s


def clean_text(s: str) -> str:
    s = _ud.normalize('NFC', str(s or '')).replace('\x00', '')
    s = s.replace('\r\n', '\n').replace('\r', '\n')
    s = '\n'.join(re.sub(r'[ \t]+', ' ', line).strip() for line in s.split('\n'))
    return re.sub(r'\n{3,}', '\n\n', s).strip()


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def stable_id(prefix: str, *parts: str, size: int = 20) -> str:
    raw = '||'.join(nfc(x) for x in parts).encode('utf-8')
    return f'{prefix}_{hashlib.sha256(raw).hexdigest()[:size]}'


HEADING_RE = re.compile(
    r'^(?:제?\s*[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+(?:\.|\s)|제?\s*\d+\s*[장절편.]|\d+(?:\.\d+){0,3}\s*[.)]?\s+'
    r'|[가-힣]\s*[.)]\s+|[【\[].{1,60}[】\]])'
)
LIST_RE = re.compile(r'^(?:[-–—•·※○ㅇ◇◆□■▶▷]|\(?\d+\)|[가-힣]\))\s*')


def classify_text(text: str) -> str:
    t = text.strip()
    if len(t) <= 120 and HEADING_RE.match(t):
        return 'heading'
    if LIST_RE.match(t):
        return 'list_item'
    return 'paragraph'


def section_level(text: str) -> int:
    t = text.strip()
    if re.match(r'^(?:제?\s*[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+|제?\s*\d+\s*장)', t):
        return 1
    m = re.match(r'^(\d+(?:\.\d+)*)', t)
    return min(4, m.group(1).count('.') + 2) if m else 2


def attach_context(elements: list) -> list:
    stack = []
    for i, e in enumerate(elements):
        if e['element_type'] == 'heading':
            level = section_level(e['text'])
            stack = stack[:level - 1] + [e['text']]
        e['section_path'] = stack.copy()
        e['header_level'] = section_level(e['text']) if e['element_type'] == 'heading' else None
        e['order_index'] = i
        e['prev_element_id'] = elements[i - 1]['element_id'] if i else None
        e['next_element_id'] = elements[i + 1]['element_id'] if i + 1 < len(elements) else None
    return elements

In [ ]:
# data_list.csv 로드: 실제 파일명은 NFD, CSV 파일명 컬럼은 NFC인 경우가 있고,
# Kaggle 업로드 문제로 파일명에서 괄호/대괄호 등 특수문자를 제거한 버전을 올렸을 수도 있어
# 두 경우 다 매칭되도록 CSV 쪽 파일명에도 동일한 정규화를 적용합니다.
# nfc()는 앞의 "공통 유틸" 셀에서 정의됩니다.
import pandas as pd


def sanitize_filename_for_match(name: str) -> str:
    p = Path(nfc(str(name)))
    base = re.sub(r'[()\[\]（）「」]', '', p.stem)
    base = base.replace('&', ' and ').replace('㈜', '')
    base = re.sub(r'\s+', ' ', base).strip()
    return base + p.suffix


meta_df = None
filename_to_row = {}
if csv_candidates:
    meta_df = pd.read_csv(csv_candidates[0])
    meta_df['파일명_match'] = meta_df['파일명'].map(sanitize_filename_for_match)
    for _, row in meta_df.iterrows():
        filename_to_row[row['파일명_match']] = row

CSV_FIELD_MAP = {
    '공고번호': '공고 번호', '공고차수': '공고 차수', '사업명': '사업명', '사업금액': '사업 금액',
    '발주기관': '발주 기관', '공개일자': '공개 일자', '입찰시작일': '입찰 참여 시작일',
    '입찰마감일': '입찰 참여 마감일', '사업요약': '사업 요약',
}

def lookup_csv_meta(path: Path) -> dict:
    row = filename_to_row.get(sanitize_filename_for_match(path.name))
    if row is None:
        return {}
    return {out_key: row.get(csv_key) for out_key, csv_key in CSV_FIELD_MAP.items()}

matched = sum(bool(lookup_csv_meta(p)) for p in source_files)
print(f'CSV 메타데이터 매칭: {matched}/{len(source_files)}')
if meta_df is not None and matched < len(source_files):
    missing = [p.name for p in source_files if not lookup_csv_meta(p)][:5]
    print('매칭 실패 샘플(최대 5개):', missing)

## 1단계: 문서 로딩 및 구조 파싱 (Extraction)

`hwp-hwpx-parser`(HWP/HWPX)와 `pymupdf`(PDF)로 원문을 파싱하되, 제목/표/이미지가 원문에 등장한
**바로 그 위치**를 보존하는 순서가 있는 요소(`elements`) 리스트로 복원합니다.

In [ ]:
# hwp-hwpx-parser 1.0.0 실측 버그 수정 (몽키패치)
#
# HWP5Reader._extract_hyperlink_texts_from_para()의 원본 소스(라이브러리를 직접 설치해 확인)는
# 문단 레코드를 순회하다가 code==0x03(컨트롤 문자)을 만났을 때, 남은 바이트가 6바이트 미만이면
# (i + 6 > len(para_data)) ctrl_id를 읽을 수 없어 그 분기를 스킵하는데, 정작 이 경우를 처리하는
# else 분기가 원본에 아예 없어서 i가 전혀 전진하지 않는다. 바깥 if code==0x03이 이미 참이었으므로
# 이어지는 elif들도 전부 건너뛰어 i가 영원히 고정된다 -> CPU 100%로 영원히 도는 무한 루프.
#
# 이 메서드는 extract_text()의 기본 호출 경로(extract_text -> _extract_section_text ->
# _collect_hyperlink_texts -> _extract_hyperlink_texts_from_para)에 포함되어 있어
# extract_text_with_notes()를 따로 쓰지 않아도 걸린다. 실측(2026-08-24): hwp 96건 중 96건 전부가
# 이 버그로 타임아웃. 아래 패치는 원본과 동일한 로직에 누락된 else(i += 2, 같은 함수의 다른 분기와
# 동일한 최소 전진폭)만 추가한 것으로, 격리 환경에서 원본은 무한 루프/패치본은 즉시 종료로 검증했다.
import struct

import hwp_hwpx_parser.hwp5 as _hwp5mod

_CTRL_ID_HYPERLINK = _hwp5mod.CTRL_ID_HYPERLINK


def _patched_extract_hyperlink_texts_from_para(self, para_data):
    hyperlink_texts = []
    i = 0
    while i < len(para_data) - 1:
        code = struct.unpack_from('<H', para_data, i)[0]
        if code == 0x03:
            if i + 6 <= len(para_data):
                ctrl_id = struct.unpack_from('<I', para_data, i + 2)[0]
                if ctrl_id == _CTRL_ID_HYPERLINK:
                    text_start = i + 14
                    text_chars = []
                    j = text_start
                    while j < len(para_data) - 1:
                        c = struct.unpack_from('<H', para_data, j)[0]
                        if c == 0x04:
                            break
                        elif c == 0x03:
                            j += 2
                        elif 0x20 <= c < 0x10000:
                            text_chars.append(chr(c))
                            j += 2
                        else:
                            j += 2
                    if text_chars:
                        hyperlink_texts.append(''.join(text_chars))
                    i = j
                    continue
                else:
                    i += 14
                    continue
            else:
                # [PATCH] 원본에 없던 else. 남은 바이트가 6바이트 미만이면 컨트롤 헤더를 더 읽을 수
                # 없으므로 최소 폭만큼 전진시켜 무한 루프를 막는다.
                i += 2
                continue
        elif code == 0x04:
            i += 10
        elif code < 32:
            if code in (11, 12):
                i += 10
            elif 15 <= code <= 23:
                i += 14
            else:
                i += 2
        else:
            i += 2
    return hyperlink_texts


_hwp5mod.HWP5Reader._extract_hyperlink_texts_from_para = _patched_extract_hyperlink_texts_from_para
print('[patch] HWP5Reader._extract_hyperlink_texts_from_para 무한 루프 버그 패치 적용 완료')

In [ ]:
# 이미지 필터링(로고/장식 이미지 제외) + OCR 유틸
# 실측(2026-08-25, BioIN 문서 등): 큰 이미지라도 빈 서식 테두리는 엣지밀도가 낮아 제외되고,
# 작지만 실제 서명/직인이 찍힌 스캔 이미지는 엣지밀도가 높아 통과함 -> 크기보다 엣지밀도가 유효한 기준.
import io

import numpy as np
import pytesseract
from PIL import Image, ImageFilter

MIN_W, MIN_H = 40, 40
MAX_ASPECT = 15.0
MIN_EDGE_DENSITY = 0.015

_ocr_cache: dict = {}


def edge_density(im: Image.Image) -> float:
    edges = im.convert('L').filter(ImageFilter.FIND_EDGES)
    arr = np.asarray(edges, dtype=np.float32)
    return float((arr > 30).mean())


def image_hash(im: Image.Image) -> str:
    small = im.convert('L').resize((8, 8))
    arr = np.asarray(small, dtype=np.float32)
    bits = arr > arr.mean()
    return ''.join('1' if b else '0' for b in bits.flatten())


def should_ocr(im: Image.Image):
    w, h = im.size
    if w < MIN_W or h < MIN_H:
        return False, 'too_small'
    aspect = max(w, h) / max(1, min(w, h))
    if aspect > MAX_ASPECT:
        return False, 'decorative_aspect'
    if edge_density(im) < MIN_EDGE_DENSITY:
        return False, 'low_edge_density'
    return True, 'ok'


def ocr_image_bytes(data: bytes):
    try:
        im = Image.open(io.BytesIO(data)).convert('RGB')
    except Exception as e:
        return '', (None, None), f'error:decode:{type(e).__name__}'
    h = image_hash(im)
    if h in _ocr_cache:
        text, status = _ocr_cache[h]
        return text, im.size, status
    ok, reason = should_ocr(im)
    if not ok:
        text, status = '', f'skipped:{reason}'
    else:
        try:
            text = clean_text(pytesseract.image_to_string(im, lang='kor+eng', config='--psm 6'))
            status = 'ok'
        except Exception as e:
            text, status = '', f'error:ocr:{type(e).__name__}'
    _ocr_cache[h] = (text, status)
    return text, im.size, status

In [ ]:
# 위치 보존 파서 유틸: hwp-hwpx-parser의 extract_text(options)는 표=Markdown, 이미지=[IMAGE: 파일명]
# 마커를 원문 등장 위치에 그대로 인라인으로 끼워 넣은 문자열 하나를 반환한다(라이브러리 소스로 확인:
# 표는 paragraphs.append(table_text)로 그 자리에서 바로 삽입되고, 이미지는 문단 텍스트 디코딩 중
# 컨트롤 문자를 만나면 문단 "중간에" 마커 문자열이 바로 끼워 넣어진다 - 마커가 항상 줄 전체를
# 차지하는 것은 아니므로 줄 단위가 아니라 정규식으로 문단 내부를 잘라내야 한다).
import re as _re

TABLE_LINE_RE = _re.compile(r'^\|.*\|$')
IMAGE_MARKER_RE = _re.compile(r'\[IMAGE(?::\s*([^\]]*))?\]')


def is_table_unit(unit: str) -> bool:
    lines = [ln.strip() for ln in unit.split('\n') if ln.strip()]
    return bool(lines) and all(TABLE_LINE_RE.match(ln) for ln in lines)


def split_into_raw_elements(raw_text: str, paragraph_separator: str = '\n\n'):
    '''paragraph_separator로 나눈 단위를 table / image / text 로 태깅해 순서대로 반환한다.'''
    out = []
    for unit in raw_text.split(paragraph_separator):
        unit = unit.strip('\n')
        if not unit.strip():
            continue
        if is_table_unit(unit):
            out.append(('table', unit, None))
            continue
        last = 0
        found_image = False
        for m in IMAGE_MARKER_RE.finditer(unit):
            found_image = True
            before = unit[last:m.start()]
            if before.strip():
                out.append(('text', before, None))
            out.append(('image', m.group(0), m.group(1)))
            last = m.end()
        if found_image:
            tail = unit[last:]
            if tail.strip():
                out.append(('text', tail, None))
        else:
            out.append(('text', unit, None))
    return out


def markdown_to_rows(md: str):
    rows = []
    for line in md.split('\n'):
        line = line.strip()
        if not line.startswith('|'):
            continue
        cells = [c.strip() for c in line.strip('|').split('|')]
        if all(_re.fullmatch(r':?-{2,}:?', c) for c in cells):
            continue
        rows.append(cells)
    return rows


def rows_to_markdown(rows):
    if not rows:
        return ''
    lines = []
    for i, row in enumerate(rows):
        clean_row = [str(c or '').replace('\n', ' ').replace('|', '\\|').strip() for c in row]
        lines.append('| ' + ' | '.join(clean_row) + ' |')
        if i == 0:
            lines.append('| ' + ' | '.join(['---'] * len(clean_row)) + ' |')
    return '\n'.join(lines)

In [ ]:
# HWP/HWPX 파서: Reader.extract_text_with_notes()를 한 번만 호출해 본문+각주+하이퍼링크를 함께
# 얻고(옵션이 같으면 get_tables()를 별도로 다시 호출할 필요가 없다 - 표는 이미 본문에 Markdown으로
# 인라인되어 있으므로 그 자리에서 직접 파싱한다), get_images()로 실제 이미지 바이트만 추가로 받는다.
from hwp_hwpx_parser import ExtractOptions, ImageMarkerStyle, Reader, TableStyle

HWP_OPTIONS = ExtractOptions(
    table_style=TableStyle.MARKDOWN,
    image_marker=ImageMarkerStyle.WITH_NAME,
    paragraph_separator='\n\n',
    line_separator='\n',
)


def parse_hwp(path: Path, canonical_id: str, image_dir: Path):
    elements, tables, cells, images = [], [], [], []
    with Reader(path) as reader:
        result = reader.extract_text_with_notes(HWP_OPTIONS)
        raw_images = reader.get_images()

    image_by_name = {}
    for ii, img in enumerate(raw_images):
        name = nfc(img.filename or f'image_{ii:03d}.{img.format}')
        ext = img.format if img.format != 'unknown' else 'bin'
        saved = image_dir / f'{canonical_id}_{ii:04d}.{ext}'
        saved.write_bytes(img.data)
        ocr_text, (w, h), status = ocr_image_bytes(img.data)
        rec = {
            'image_id': stable_id('img', canonical_id, str(ii)), 'canonical_doc_id': canonical_id,
            'image_index': ii, 'source_name': name, 'saved_path': str(saved.relative_to(OUTPUT_ROOT)),
            'format': img.format, 'width': w, 'height': h, 'ocr_text': ocr_text, 'ocr_status': status,
        }
        images.append(rec)
        image_by_name[name] = rec

    for kind, raw, image_name in split_into_raw_elements(result.text, HWP_OPTIONS.paragraph_separator):
        idx = len(elements)
        eid = stable_id('el', canonical_id, str(idx))
        if kind == 'table':
            rows = markdown_to_rows(raw)
            table_id = stable_id('tbl', canonical_id, str(len(tables)))
            tables.append({
                'table_id': table_id, 'canonical_doc_id': canonical_id, 'table_index': len(tables),
                'row_count': len(rows), 'col_count': len(rows[0]) if rows else 0,
                'markdown': clean_text(raw), 'rows': rows,
            })
            for ri, row in enumerate(rows):
                for ci, value in enumerate(row):
                    cells.append({
                        'cell_id': stable_id('cell', table_id, str(ri), str(ci)), 'table_id': table_id,
                        'canonical_doc_id': canonical_id, 'row_index': ri, 'col_index': ci,
                        'text': clean_text(value),
                    })
            elements.append({
                'element_id': eid, 'canonical_doc_id': canonical_id, 'element_type': 'table',
                'text': clean_text(raw), 'table_id': table_id, 'image_id': None,
                'evidence': {'source_file': nfc(path.name), 'parser': 'hwp-hwpx-parser', 'logical_index': idx},
            })
        elif kind == 'image':
            rec = image_by_name.get(nfc(image_name or ''))
            image_id = rec['image_id'] if rec else None
            text = rec['ocr_text'] if rec else ''
            elements.append({
                'element_id': eid, 'canonical_doc_id': canonical_id, 'element_type': 'image',
                'text': text, 'table_id': None, 'image_id': image_id,
                'evidence': {'source_file': nfc(path.name), 'parser': 'hwp-hwpx-parser', 'logical_index': idx,
                             'image_marker_name': nfc(image_name or '')},
            })
        else:
            text = clean_text(raw)
            if not text:
                continue
            elements.append({
                'element_id': eid, 'canonical_doc_id': canonical_id, 'element_type': classify_text(text),
                'text': text, 'table_id': None, 'image_id': None,
                'evidence': {'source_file': nfc(path.name), 'parser': 'hwp-hwpx-parser', 'logical_index': idx},
            })

    extras = {
        'footnotes': [vars(x) for x in result.footnotes], 'endnotes': [vars(x) for x in result.endnotes],
        'hyperlinks': [list(x) for x in result.hyperlinks], 'memos': [vars(x) for x in result.memos],
    }
    return attach_context(elements), tables, cells, images, extras, result.text

In [ ]:
# PDF 파서: pymupdf 블록 좌표(페이지+bbox)를 근거(evidence)로 보존하고, find_tables()로 표를 감지해
# Markdown으로 구조화한다. 표 영역과 겹치는 일반 텍스트 블록은 중복 수록하지 않는다.
import fitz
import tempfile

fitz.TOOLS.mupdf_display_errors(False)  # invalid key in dict 등 MuPDF 내부 경고 stdout 억제(파싱 결과엔 영향 없음)


def bbox_overlap_ratio(inner, outer) -> float:
    x0, y0 = max(inner[0], outer[0]), max(inner[1], outer[1])
    x1, y1 = min(inner[2], outer[2]), min(inner[3], outer[3])
    if x1 <= x0 or y1 <= y0:
        return 0.0
    inter = (x1 - x0) * (y1 - y0)
    area = max(1e-6, (inner[2] - inner[0]) * (inner[3] - inner[1]))
    return inter / area


def parse_pdf(path: Path, canonical_id: str, image_dir: Path):
    elements, tables, cells, images = [], [], [], []
    full_text_parts = []
    doc = fitz.open(path)
    try:
        repaired_path = Path(tempfile.gettempdir()) / f'{canonical_id}_repaired.pdf'
        doc.save(str(repaired_path), garbage=4, clean=True, deflate=True)
        doc.close()
        doc = fitz.open(repaired_path)
    except Exception:
        pass  # 복구 저장 실패 시 원본 doc(이미 열려 있음)을 그대로 사용
    for page_no, page in enumerate(doc, start=1):
        try:
            table_finder = page.find_tables()
            page_tables = list(table_finder.tables) if table_finder else []
        except ParseTimeout:
            raise  # 문서 전체 타임아웃 예산 소진: 남은 페이지를 무방비로 진행하지 않고 상위로 전파
        except Exception:
            page_tables = []

        table_entries = []
        for ti, tbl in enumerate(page_tables):
            rows = tbl.extract()
            if not rows or not any(any(c for c in row) for row in rows):
                continue
            table_id = stable_id('tbl', canonical_id, str(page_no), str(ti))
            md = rows_to_markdown(rows)
            tables.append({
                'table_id': table_id, 'canonical_doc_id': canonical_id, 'table_index': len(tables),
                'row_count': len(rows), 'col_count': len(rows[0]) if rows else 0, 'markdown': md, 'rows': rows,
            })
            for ri, row in enumerate(rows):
                for ci, value in enumerate(row):
                    cells.append({
                        'cell_id': stable_id('cell', table_id, str(ri), str(ci)), 'table_id': table_id,
                        'canonical_doc_id': canonical_id, 'row_index': ri, 'col_index': ci,
                        'text': clean_text(value or ''),
                    })
            table_entries.append({'bbox': tuple(tbl.bbox), 'table_id': table_id, 'markdown': md})
            full_text_parts.append(md)

        page_blocks = sorted(page.get_text('blocks'), key=lambda b: (round(b[1], 1), round(b[0], 1)))
        for bi, b in enumerate(page_blocks):
            text = clean_text(b[4])
            if not text:
                continue
            block_bbox = tuple(b[:4])
            if any(bbox_overlap_ratio(block_bbox, t['bbox']) > 0.5 for t in table_entries):
                continue
            idx = len(elements)
            elements.append({
                'element_id': stable_id('el', canonical_id, str(idx)), 'canonical_doc_id': canonical_id,
                'element_type': classify_text(text), 'text': text, 'table_id': None, 'image_id': None,
                'evidence': {'source_file': nfc(path.name), 'parser': 'pymupdf', 'page': page_no,
                             'bbox': [round(float(x), 2) for x in block_bbox], 'page_block_index': bi},
            })
            full_text_parts.append(text)

        for te in table_entries:
            idx = len(elements)
            elements.append({
                'element_id': stable_id('el', canonical_id, str(idx)), 'canonical_doc_id': canonical_id,
                'element_type': 'table', 'text': te['markdown'], 'table_id': te['table_id'], 'image_id': None,
                'evidence': {'source_file': nfc(path.name), 'parser': 'pymupdf.find_tables', 'page': page_no,
                             'bbox': [round(float(x), 2) for x in te['bbox']]},
            })

        for ii, info in enumerate(page.get_images(full=True)):
            xref = info[0]
            img_data = doc.extract_image(xref)
            payload, ext = img_data['image'], img_data.get('ext', 'bin')
            global_i = len(images)
            saved = image_dir / f'{canonical_id}_{global_i:04d}.{ext}'
            saved.write_bytes(payload)
            ocr_text, (w, h), status = ocr_image_bytes(payload)
            image_id = stable_id('img', canonical_id, str(global_i))
            images.append({
                'image_id': image_id, 'canonical_doc_id': canonical_id, 'image_index': global_i,
                'source_name': f'page_{page_no}_image_{ii}', 'saved_path': str(saved.relative_to(OUTPUT_ROOT)),
                'format': ext, 'width': w, 'height': h, 'ocr_text': ocr_text, 'ocr_status': status,
                'page': page_no, 'xref': xref,
            })
            idx = len(elements)
            elements.append({
                'element_id': stable_id('el', canonical_id, str(idx)), 'canonical_doc_id': canonical_id,
                'element_type': 'image', 'text': ocr_text, 'table_id': None, 'image_id': image_id,
                'evidence': {'source_file': nfc(path.name), 'parser': 'pymupdf+pytesseract', 'page': page_no,
                             'xref': xref},
            })
            if ocr_text:
                full_text_parts.append(ocr_text)
    doc.close()

    def sort_key(e):
        ev = e['evidence']
        page = ev.get('page', 0)
        y = ev.get('bbox', [0, 10 ** 9])[1]
        return (page, y, e['element_id'])

    elements.sort(key=sort_key)
    return attach_context(elements), tables, cells, images, {}, '\n\n'.join(full_text_parts)

In [ ]:
# 문서 단위 파싱 실행: SHA-256 전체 바이트 해시로 canonical_doc_id를 배정하고(원문이 완전히 같은
# 파일은 파일명이 달라도 같은 문서로 취급), CSV 메타데이터를 조인한다. 패치로 무한 루프의 근본 원인은
# 고쳤지만, 아직 발견되지 않은 다른 edge case에 대비해 문서 1건당 signal 기반 하드 타임아웃을 안전망으로
# 둔다(신호 기반 타임아웃은 CUDA/GPU 프로세스를 쓰지 않으므로 fork+CUDA 데드락 위험이 없다).
import signal
import traceback
from collections import Counter, defaultdict

from tqdm.auto import tqdm

PARSE_TIMEOUT_SECONDS = 60


class ParseTimeout(Exception):
    pass


def _timeout_handler(signum, frame):
    raise ParseTimeout(f'파싱이 {PARSE_TIMEOUT_SECONDS}초를 넘겨 중단했습니다')


def parse_with_timeout(fn, *args):
    has_alarm = hasattr(signal, 'SIGALRM')
    if not has_alarm:
        return fn(*args)
    old_handler = signal.signal(signal.SIGALRM, _timeout_handler)
    signal.alarm(PARSE_TIMEOUT_SECONDS)
    try:
        return fn(*args)
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)


file_records = []
for path in source_files:
    payload = path.read_bytes()
    digest = sha256_bytes(payload)
    file_records.append({'path': path, 'file_sha256': digest, 'canonical_doc_id': f'doc_{digest[:20]}',
                          'file_size': len(payload)})
hash_counts = Counter(x['file_sha256'] for x in file_records)
print('물리 원본 중복 그룹:', sum(v > 1 for v in hash_counts.values()),
      '| 중복 파일 수:', sum(v for v in hash_counts.values() if v > 1))

documents, all_elements, all_tables, all_cells, all_images, errors = [], [], [], [], [], []
parsed_cache = {}
for f in tqdm(file_records, desc='RFP 파싱'):
    path, cid = f['path'], f['canonical_doc_id']
    try:
        if cid not in parsed_cache:
            parse_fn = parse_pdf if path.suffix.lower() == '.pdf' else parse_hwp
            result = parse_with_timeout(parse_fn, path, cid, IMAGE_DIR)
            parsed_cache[cid] = result
            elements, tables, cells, images, extras, raw_text = result
            all_elements.extend(elements)
            all_tables.extend(tables)
            all_cells.extend(cells)
            all_images.extend(images)
        else:
            elements, tables, cells, images, extras, raw_text = parsed_cache[cid]

        meta = lookup_csv_meta(path)
        documents.append({
            'source_instance_id': stable_id('src', str(path.relative_to(INPUT_ROOT))),
            'canonical_doc_id': cid, 'source_path': nfc(str(path.relative_to(INPUT_ROOT))),
            'source_filename': nfc(path.name), 'extension': path.suffix.lower(),
            'file_sha256': f['file_sha256'], 'file_size': f['file_size'],
            'is_exact_duplicate': hash_counts[f['file_sha256']] > 1,
            'exact_duplicate_count': hash_counts[f['file_sha256']],
            'character_count': len(raw_text), 'element_count': len(elements),
            'table_count': len(tables), 'image_count': len(images), 'extras': extras,
            'csv_metadata_matched': bool(meta), **meta,
            'schema_version': SCHEMA_VERSION, 'parsed_at': PARSED_AT,
        })
    except Exception as e:
        errors.append({
            'source_path': nfc(str(path.relative_to(INPUT_ROOT))), 'canonical_doc_id': cid,
            'error_type': type(e).__name__, 'message': str(e), 'traceback': traceback.format_exc(limit=5),
        })

canonical_ids = set(d['canonical_doc_id'] for d in documents)
print(f'성공 {len(documents)} / 입력 {len(source_files)}, 오류 {len(errors)}, canonical 문서 {len(canonical_ids)}')
if errors:
    print('오류 샘플:', errors[:3])

## 2단계: 노이즈 제거 및 정제 (Cleaning)

머리말/꼬리말, 페이지 번호, HWP 제어 아티팩트, 연속 공백/줄바꿈을 제거합니다. 무엇을 왜 지웠는지는
`text_raw`(원문)를 남기고 `artifact_logs.jsonl`에 감사 로그로 기록해 검증 가능하게 합니다.

In [ ]:
# 제어 아티팩트/머리말·꼬리말/페이지번호 정제 (원문 보존 + 감사 로그)
# MOJIBAKE_RE: U+FFFD 외에, 이 데이터셋 실측에서 반복 확인된 HWP 목차 점선 리더 깨짐 패턴도 포함한다.
# 목차의 "......." 점선이 심볼 폰트로 인코딩돼 있어 실제로는 정상 유니코드 문자로 디코딩되는데,
# 앞 글자는 "螨/砼/牠/沈..."처럼 줄마다 달라지고 뒤 글자만 항상 'ȃ'(U+0203, 한국어 문서에 정상적으로
# 등장할 일이 없는 라틴 확장 문자)로 고정되는 실측 패턴을 확인했다 - 즉 앞 글자 값이 아니라 뒤에 오는
# 고정 문자 'ȃ'가 이 아티팩트의 진짜 지문이므로, "임의의 한 글자 + ȃ" 2글자 단위를 통째로 제거한다.
MOJIBAKE_RE = re.compile(r'�+|(?:.ȃ)+')
PAGE_NO_RE = re.compile(r'^\s*(?:[-–—]?\s*\d{1,4}\s*[-–—]?|page\s*\d{1,4})\s*$', re.I)
DOT_LEADER_RE = re.compile(r'[.·ㆍ…]{4,}\s*\d{1,4}\s*$')
ALLOWED_CONTROLS = {'\n', '\t'}


def clean_with_audit(text: str):
    raw = _ud.normalize('NFC', text)
    removed, kept = [], []
    for pos, ch in enumerate(raw):
        category = _ud.category(ch)
        if category in {'Cc', 'Cf', 'Cs', 'Co', 'Cn'} and ch not in ALLOWED_CONTROLS:
            removed.append({'position': pos, 'text': repr(ch), 'reason': f'unicode_{category}'})
        else:
            kept.append(ch)
    value = ''.join(kept)

    def drop_mojibake(m):
        removed.append({'position': m.start(), 'text': m.group(0), 'reason': 'decode_artifact'})
        return ' '

    value = MOJIBAKE_RE.sub(drop_mojibake, value)
    value = '\n'.join(DOT_LEADER_RE.sub('', line) for line in value.splitlines())
    value = clean_text(value)
    ratio = round(max(0.0, (len(raw) - len(value)) / max(1, len(raw))), 6)
    return value, removed, ratio


artifact_logs = []
for e in all_elements:
    e['text_raw'] = e['text']
    e['text'], removed, ratio = clean_with_audit(e['text_raw'])
    e['artifact_ratio'] = ratio
    e['needs_review'] = ratio > 0.05 or len(removed) > 10
    if removed:
        artifact_logs.append({'element_id': e['element_id'], 'canonical_doc_id': e['canonical_doc_id'],
                               'removed_count': len(removed), 'artifact_ratio': ratio,
                               'removed': removed[:100]})
    if not e['text']:
        e['noise_label'] = 'empty_after_cleaning'
    elif PAGE_NO_RE.match(e['text']):
        e['noise_label'] = 'page_number'
    else:
        bbox = e['evidence'].get('bbox')
        if bbox and bbox[1] < 45:
            e['noise_label'] = 'pdf_header_candidate'
        elif bbox and bbox[1] > 790:
            e['noise_label'] = 'pdf_footer_candidate'
        else:
            e['noise_label'] = None

# 동일 문서 안에서 3회 이상 반복되는 짧은 문구(머리말/꼬리말 후보)는 삭제하지 않고 표시만 한다
short_counter = defaultdict(Counter)
for e in all_elements:
    if 3 <= len(e['text']) <= 100:
        short_counter[e['canonical_doc_id']][e['text']] += 1
NOISE_EXCLUDE = {'page_number', 'empty_after_cleaning', 'pdf_header_candidate', 'pdf_footer_candidate'}
for e in all_elements:
    if e['noise_label'] is None and e['element_type'] == 'paragraph' \
            and short_counter[e['canonical_doc_id']][e['text']] >= 3:
        e['noise_label'] = 'repeated_short_text'
    e['include_in_analysis'] = e['noise_label'] not in (NOISE_EXCLUDE | {'repeated_short_text'})

print('정제 요소:', len(all_elements), '| 아티팩트 감지:', len(artifact_logs),
      '| 검토 필요:', sum(e['needs_review'] for e in all_elements),
      '| 분석 제외:', sum(not e['include_in_analysis'] for e in all_elements))

## 3단계: 표/이미지/메타데이터 구조화 (Multimodal Processing)

표는 이미 Markdown으로, 이미지는 OCR 텍스트로 구조화되어 있습니다. 문서 메타데이터는 `data_list.csv`
기준(발주기관/사업명/사업금액/공고일자)을 그대로 사용합니다 - 본문에서 다시 추측하지 않습니다.

In [ ]:
elements_by_doc = defaultdict(list)
for e in all_elements:
    if e['include_in_analysis']:
        elements_by_doc[e['canonical_doc_id']].append(e)
for cid in elements_by_doc:
    elements_by_doc[cid].sort(key=lambda x: x['order_index'])

doc_meta_by_id = {d['canonical_doc_id']: d for d in documents}
for t in all_tables:
    t['content_type'] = 'table'
    t['serialization'] = 'markdown'
for im in all_images:
    im['content_type'] = 'image_ocr'
    im['caption_candidate'] = im['ocr_text'][:300] if im['ocr_text'] else None

print('메타데이터 매칭 문서:', sum(d['csv_metadata_matched'] for d in documents), '/', len(documents))
print('표:', len(all_tables), '| 이미지:', len(all_images),
      '| OCR 텍스트 있는 이미지:', sum(bool(x['ocr_text']) for x in all_images))

## 4단계: 의미 단위 청킹 (Chunking)

섹션/문단 경계를 우선하고, 300~800 토큰(BGE-m3 토크나이저 기준)을 목표로 청킹합니다. 표/이미지는
독립 청크로 분리하되 앞뒤 문맥 요소를 함께 묶어 근거를 유지합니다.

In [ ]:
from transformers import AutoTokenizer

DENSE_MODEL = 'BAAI/bge-m3'  # GPU/메모리가 부족하면 'BM-K/KoSimCSE-roberta-multitask'로 교체
dense_tokenizer = AutoTokenizer.from_pretrained(DENSE_MODEL)
MIN_TOKENS, TARGET_TOKENS, MAX_TOKENS, OVERLAP_TOKENS = 300, 600, 800, 60


def token_count(text: str) -> int:
    return len(dense_tokenizer.encode(text, add_special_tokens=False))


def split_long_text(text: str, max_tokens: int = MAX_TOKENS):
    ids = dense_tokenizer.encode(text, add_special_tokens=False)
    parts, start = [], 0
    while start < len(ids):
        end = min(len(ids), start + max_tokens)
        parts.append(dense_tokenizer.decode(ids[start:end], skip_special_tokens=True).strip())
        if end == len(ids):
            break
        start = max(start + 1, end - OVERLAP_TOKENS)
    return [p for p in parts if p]


chunks = []
chunk_seq = defaultdict(int)


def doc_title(cid):
    meta = doc_meta_by_id.get(cid, {})
    return meta.get('사업명') or meta.get('source_filename')


def emit_chunk(cid, buf, kind='text', forced_text=None):
    text = (forced_text if forced_text is not None else '\n\n'.join(x['text'] for x in buf if x['text'])).strip()
    if not text:
        return
    for part in split_long_text(text):
        idx = chunk_seq[cid]
        chunk_seq[cid] += 1
        meta = doc_meta_by_id.get(cid, {})
        chunks.append({
            'chunk_id': stable_id('chk', cid, str(idx)), 'text': part,
            'metadata': {
                'canonical_doc_id': cid, 'chunk_index': idx, 'chunk_type': kind,
                'document_title': doc_title(cid), '발주기관': meta.get('발주기관'),
                '공개일자': meta.get('공개일자'), '사업금액': meta.get('사업금액'),
                'section_path': buf[-1].get('section_path', []) if buf else [],
                'header_level': buf[-1].get('header_level') if buf else None,
                'element_ids': [x['element_id'] for x in buf],
                'order_index_start': min((x['order_index'] for x in buf), default=None),
                'order_index_end': max((x['order_index'] for x in buf), default=None),
                'source_filenames': sorted(set(x['evidence']['source_file'] for x in buf)),
                'needs_review': any(x['needs_review'] for x in buf),
            },
            'token_count': token_count(part),
        })


for cid, doc_elements in tqdm(elements_by_doc.items(), desc='청킹'):
    lookup = {x['element_id']: x for x in doc_elements}
    buf, size = [], 0
    for e in doc_elements:
        special = e['element_type'] in {'table', 'image'}
        et = token_count(e['text'])
        if e['element_type'] == 'heading' and size >= MIN_TOKENS:
            emit_chunk(cid, buf)
            buf, size = [], 0
        if special:
            if buf:
                emit_chunk(cid, buf)
                buf, size = [], 0
            context = [lookup[x] for x in (e.get('prev_element_id'),) if x in lookup] + [e] \
                + [lookup[x] for x in (e.get('next_element_id'),) if x in lookup]
            emit_chunk(cid, context, e['element_type'])
            continue
        if buf and size + et > MAX_TOKENS:
            emit_chunk(cid, buf)
            buf, size = [], 0
        buf.append(e)
        size += et
        if size >= TARGET_TOKENS:
            emit_chunk(cid, buf)
            buf, size = [], 0
    if buf:
        emit_chunk(cid, buf)

for c in chunks:
    m = c['metadata']
    c['retrieval_text'] = ' '.join(filter(None, [
        m.get('document_title'), m.get('발주기관'), ' '.join(m.get('section_path', [])), c['text'],
    ]))

token_counts = [c['token_count'] for c in chunks]
print('청크:', len(chunks), '| 토큰 중앙값:', int(np.median(token_counts)) if chunks else 0,
      '| 800 초과:', sum(t > 800 for t in token_counts))

## 5단계: 사용자 정의 어휘사전(Custom Dictionary) 반영

기관명/사업명(CSV)과 청크에서 자주 등장하는 명사/약어를 후보로 모아 Kiwi 사용자 사전에 등록합니다.
긴 용어부터 등록해야 짧은 하위 형태소로 먼저 잘리는 과분절을 피할 수 있습니다.

In [ ]:
from kiwipiepy import Kiwi

kiwi = Kiwi()
STOPWORDS = {'사업', '시스템', '제안', '요청', '관련', '내용', '경우', '기관', '업무', '통해', '위해',
             '대한', '따라', '등', '및', '수', '함'}
ACRONYM_RE = re.compile(r'\b[A-Z][A-Z0-9-]{1,15}\b')
ORG_NAME_RE = re.compile(r'[가-힣A-Za-z0-9·-]{2,}(?:대학교|연구원|공사|공단|재단|센터|협회|진흥원)')

term_tf, term_docs, term_examples = Counter(), defaultdict(set), defaultdict(list)
seed_terms = set()
for d in documents:
    for field in ('발주기관', '사업명'):
        seed_terms.update(ORG_NAME_RE.findall(str(d.get(field) or '')))
    seed_terms.update(ACRONYM_RE.findall(str(d.get('사업명') or '')))

for c in tqdm(chunks, desc='도메인 용어 추출'):
    terms = [t.form for t in kiwi.tokenize(c['text'])
             if t.tag.startswith(('NN', 'SL')) and len(t.form) >= 2 and t.form not in STOPWORDS]
    terms += ACRONYM_RE.findall(c['text'])
    for term in terms:
        term_tf[term] += 1
        term_docs[term].add(c['metadata']['canonical_doc_id'])
        if len(term_examples[term]) < 3:
            term_examples[term].append({'chunk_id': c['chunk_id'], 'snippet': c['text'][:180]})
for term in seed_terms:
    term_tf[term] += 3

domain_dictionary = []
for term, freq in term_tf.most_common():
    if freq < 3 or len(term) > 40:
        continue
    domain_dictionary.append({
        'term': term, 'term_normalized': term.lower(), 'frequency': freq,
        'document_frequency': len(term_docs[term]),
        'term_type': 'acronym' if ACRONYM_RE.fullmatch(term) else 'noun',
        'examples': term_examples[term], 'needs_definition_review': True,
    })

registered_terms = []
for row in sorted(domain_dictionary, key=lambda x: len(x['term']), reverse=True):
    try:
        kiwi.add_user_word(row['term'], 'NNP', 0.0)
        registered_terms.append(row['term'])
    except Exception:
        pass
print('도메인 사전 후보:', len(domain_dictionary), '| Kiwi 등록:', len(registered_terms))

## 6단계: RAG용 데이터 준비 - Sparse(BM25) + Dense(Embedding) 이중 인덱싱

In [ ]:
import json
import pickle

import faiss
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer


def tokenize_search(text: str):
    return [t.form.lower() for t in kiwi.tokenize(text) if t.tag.startswith(('NN', 'VV', 'VA', 'SL', 'SN'))]


corpus_tokens = [tokenize_search(c['retrieval_text']) for c in tqdm(chunks, desc='BM25 토큰화')]
bm25 = BM25Okapi(corpus_tokens)
with (OUTPUT_ROOT / 'bm25_index.pkl').open('wb') as f:
    pickle.dump({'bm25': bm25, 'chunk_ids': [c['chunk_id'] for c in chunks]}, f)

# Kaggle Notebook Settings > Accelerator에서 GPU(T4 등)를 켜두면 여기서 자동으로 잡아 쓴다.
# 이 노트북은 hwp 추출에 multiprocessing fork를 쓰지 않고(신호 기반 타임아웃만 사용) OCR도
# GPU가 아닌 CPU Tesseract이므로, CUDA를 켜둔 채로 실행해도 fork+CUDA 데드락 위험이 없다.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EMBED_BATCH_SIZE = 64 if DEVICE == 'cuda' else 8
print(f'Dense 임베딩 디바이스: {DEVICE} (batch_size={EMBED_BATCH_SIZE})')
if DEVICE == 'cpu':
    print('GPU가 감지되지 않았습니다. Kaggle Notebook Settings > Accelerator에서 GPU를 켜면 '
          f'{len(chunks)}개 청크 임베딩이 훨씬 빨라집니다(현재 CPU로도 동작은 합니다).')

encoder = SentenceTransformer(DENSE_MODEL, device=DEVICE)
embeddings = encoder.encode(
    [c['retrieval_text'] for c in chunks], batch_size=EMBED_BATCH_SIZE, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True,
).astype('float32')
dense_index = faiss.IndexFlatIP(embeddings.shape[1])
dense_index.add(embeddings)
faiss.write_index(dense_index, str(OUTPUT_ROOT / 'dense_index.faiss'))
np.save(OUTPUT_ROOT / 'dense_embeddings.npy', embeddings)
(OUTPUT_ROOT / 'index_config.json').write_text(json.dumps({
    'dense_model': DENSE_MODEL, 'metric': 'cosine_via_normalized_inner_product',
    'bm25_tokenizer': 'kiwipiepy', 'chunk_count': len(chunks), 'embedding_device': DEVICE,
}, ensure_ascii=False, indent=2), encoding='utf-8')


def minmax(a):
    a = np.asarray(a, dtype='float32')
    span = float(a.max() - a.min()) if len(a) else 0
    return (a - a.min()) / span if span > 1e-9 else np.zeros_like(a)


def hybrid_search(query: str, k: int = 5, alpha: float = 0.55):
    sparse = np.asarray(bm25.get_scores(tokenize_search(query)), dtype='float32')
    q = encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype('float32')
    dense = embeddings @ q[0]
    score = alpha * minmax(dense) + (1 - alpha) * minmax(sparse)
    idx = np.argsort(-score)[:k]
    return [{
        'rank': r + 1, 'score': float(score[i]), 'dense_score': float(dense[i]), 'bm25_score': float(sparse[i]),
        'chunk_id': chunks[i]['chunk_id'], 'canonical_doc_id': chunks[i]['metadata']['canonical_doc_id'],
        'text': chunks[i]['text'], 'metadata': chunks[i]['metadata'],
    } for r, i in enumerate(idx)]


print('인덱스 완료:', dense_index.ntotal, 'vectors')

## 7단계: RAG 평가 및 Benchmark 데이터셋(JSONL) 생성

`chunk_dataset.jsonl`은 `{"chunk_id", "text", "metadata"}` 형태입니다. `rag_benchmark.jsonl`은
섹션 제목+핵심어 기반으로 자동 생성한 retrieval 회귀시험용 질의이며, 사람이 질문·정답·근거를 검수하기
전에는 golden QA로 간주하지 않습니다(`needs_review=true`).

In [ ]:
benchmark = []
for c in chunks:
    if c['token_count'] < 80:
        continue
    meta = c['metadata']
    section = ' > '.join(meta['section_path'][-2:]) if meta['section_path'] else ''
    nouns = [x for x in tokenize_search(c['text']) if len(x) >= 2 and x not in STOPWORDS]
    keywords = [x for x, _ in Counter(nouns).most_common(4)]
    if not keywords:
        continue
    query = (section + ' ' + ' '.join(keywords)).strip()
    benchmark.append({
        'benchmark_id': stable_id('bench', c['chunk_id']), 'task_type': 'retrieval_probe',
        'query': query, 'answer_candidate': c['text'][:500],
        'expected_chunk_ids': [c['chunk_id']], 'expected_doc_ids': [meta['canonical_doc_id']],
        'evidence': {'chunk_id': c['chunk_id'], 'element_ids': meta['element_ids'],
                     'source_filenames': meta['source_filenames'], 'section_path': meta['section_path'],
                     'order_index_start': meta['order_index_start'], 'order_index_end': meta['order_index_end']},
        'generation_method': 'section_heading_plus_keyterms', 'needs_review': True,
    })

by_bench_doc = defaultdict(list)
for x in benchmark:
    by_bench_doc[x['expected_doc_ids'][0]].append(x)
per_doc = max(1, 1000 // max(1, len(by_bench_doc)))
balanced = []
for doc_id, rows in sorted(by_bench_doc.items()):
    balanced.extend(rows[:per_doc])
benchmark = balanced[:1000]

ranks = []
for item in tqdm(benchmark, desc='Hybrid retrieval 자체 평가'):
    hits = hybrid_search(item['query'], k=10)
    ids = [x['chunk_id'] for x in hits]
    target = item['expected_chunk_ids'][0]
    rank = (ids.index(target) + 1) if target in ids else None
    item['baseline_retrieval'] = {'rank': rank, 'top10_chunk_ids': ids}
    ranks.append(rank)

retrieval_metrics = {
    'count': len(ranks), 'recall_at_1': sum(r == 1 for r in ranks) / max(1, len(ranks)),
    'recall_at_5': sum(r is not None and r <= 5 for r in ranks) / max(1, len(ranks)),
    'recall_at_10': sum(r is not None and r <= 10 for r in ranks) / max(1, len(ranks)),
    'mrr_at_10': sum((1 / r if r else 0) for r in ranks) / max(1, len(ranks)),
}
print(json.dumps(retrieval_metrics, ensure_ascii=False, indent=2))

In [ ]:
# 저장 및 품질 검증
def write_jsonl(name, rows):
    with (OUTPUT_ROOT / name).open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, default=str) + '\n')


for name, rows in [
    ('documents.jsonl', documents), ('elements.jsonl', all_elements), ('tables.jsonl', all_tables),
    ('table_cells.jsonl', all_cells), ('images.jsonl', all_images), ('artifact_logs.jsonl', artifact_logs),
    ('chunk_dataset.jsonl', chunks), ('domain_dictionary.jsonl', domain_dictionary),
    ('rag_benchmark.jsonl', benchmark), ('errors.jsonl', errors),
]:
    write_jsonl(name, rows)

(OUTPUT_ROOT / 'kiwi_user_dictionary.txt').write_text(
    '\n'.join(f'{term}\tNNP\t0.0' for term in registered_terms), encoding='utf-8')
pd.json_normalize(documents).drop(columns=['extras'], errors='ignore').to_csv(
    OUTPUT_ROOT / 'documents.csv', index=False, encoding='utf-8-sig')
pd.json_normalize(all_elements).drop(columns=['text_raw'], errors='ignore').to_csv(
    OUTPUT_ROOT / 'elements.csv', index=False, encoding='utf-8-sig')
pd.json_normalize(chunks).to_csv(OUTPUT_ROOT / 'chunks.csv', index=False, encoding='utf-8-sig')
pd.json_normalize(domain_dictionary).drop(columns=['examples'], errors='ignore').to_csv(
    OUTPUT_ROOT / 'domain_dictionary.csv', index=False, encoding='utf-8-sig')

qa = {
    'schema_version': SCHEMA_VERSION, 'input_document_count': len(source_files),
    'parsed_source_count': len(documents), 'canonical_document_count': len(canonical_ids),
    'exact_duplicate_groups': sum(v > 1 for v in hash_counts.values()),
    'csv_metadata_matched_count': sum(d['csv_metadata_matched'] for d in documents),
    'element_count': len(all_elements), 'table_count': len(all_tables), 'table_cell_count': len(all_cells),
    'image_count': len(all_images), 'ocr_nonempty_count': sum(bool(x['ocr_text']) for x in all_images),
    'artifact_log_count': len(artifact_logs), 'review_element_count': sum(e['needs_review'] for e in all_elements),
    'chunk_count': len(chunks),
    'chunk_token_min': min((x['token_count'] for x in chunks), default=0),
    'chunk_token_median': float(np.median([x['token_count'] for x in chunks])) if chunks else 0,
    'chunk_token_max': max((x['token_count'] for x in chunks), default=0),
    'dictionary_term_count': len(domain_dictionary), 'registered_dictionary_term_count': len(registered_terms),
    'dense_model': DENSE_MODEL, 'benchmark_count': len(benchmark), 'retrieval_metrics': retrieval_metrics,
    'error_count': len(errors), 'element_type_counts': dict(Counter(x['element_type'] for x in all_elements)),
    'checks': {
        'all_elements_have_order': all(isinstance(x.get('order_index'), int) for x in all_elements),
        'all_elements_have_evidence': all(bool(x.get('evidence')) for x in all_elements),
        'all_chunks_have_metadata': all(bool(x.get('metadata')) for x in chunks),
        'chunk_token_max_800': all(x['token_count'] <= 800 for x in chunks),
        'kiwi_user_dictionary_registered': len(registered_terms) > 0,
        'duplicate_ids_collapsed': all(
            len(set(x['canonical_doc_id'] for x in documents if x['file_sha256'] == h)) == 1
            for h, c in hash_counts.items() if c > 1
        ),
    },
}
(OUTPUT_ROOT / 'quality_report.json').write_text(json.dumps(qa, ensure_ascii=False, indent=2), encoding='utf-8')

readme = f'''# RFP RAG 데이터셋

- 스키마: {SCHEMA_VERSION}
- 입력: {len(source_files)}건 / 성공: {len(documents)}건 / 오류: {len(errors)}건
- canonical 문서: {qa['canonical_document_count']}건 (CSV 메타데이터 매칭 {qa['csv_metadata_matched_count']}건)
- 요소: {len(all_elements):,} / 표: {len(all_tables):,} / 셀: {len(all_cells):,} / 이미지: {len(all_images):,} / 청크: {len(chunks):,}
- Dense 모델: {DENSE_MODEL}
- 청크 토큰: 최소 {qa['chunk_token_min']} / 중앙값 {qa['chunk_token_median']:.0f} / 최대 {qa['chunk_token_max']}
- Self-retrieval 회귀 지표: {json.dumps(retrieval_metrics, ensure_ascii=False)}

## 파이프라인
1. 문서 로딩 및 구조 파싱 (HWP 무한 루프 버그 패치 적용, 위치 보존 요소 추출)
2. 제어 아티팩트/머리말/꼬리말/페이지번호 정제 (원문 `text_raw` 보존, 감사 로그)
3. 표(Markdown)/이미지(OCR, 엣지밀도 기반 로고 필터링)/메타데이터(CSV 조인) 구조화
4. 섹션 경계 기반 300~800 토큰 의미 청킹
5. 기관명/사업명/약어 기반 도메인 사전 생성 및 Kiwi 사용자 사전 등록
6. Kiwi BM25 + BGE-m3 FAISS Dense 하이브리드 인덱스
7. 자동 생성 retrieval 회귀 질의로 Recall@K/MRR 측정, Benchmark JSONL 생성

`rag_benchmark.jsonl`은 자동 생성된 검수 후보이므로 `needs_review=true`입니다. 사람이 질문·정답·근거를
확인한 뒤 golden benchmark로 승격해야 합니다.
'''
(OUTPUT_ROOT / 'README.md').write_text(readme, encoding='utf-8')
print(json.dumps(qa, ensure_ascii=False, indent=2))
doc_preview = pd.DataFrame(documents)
preview_cols = [c for c in [
    'source_filename', 'canonical_doc_id', '발주기관', '사업명', 'is_exact_duplicate',
    'element_count', 'table_count', 'image_count',
] if c in doc_preview.columns]
display(doc_preview[preview_cols].head(10))

In [ ]:
# 최종 ZIP 다운로드
archive = shutil.make_archive('/kaggle/working/RFP_100_RAG_dataset', 'zip', OUTPUT_ROOT)
print('완료:', archive)
print('Kaggle Notebook의 Output 탭에서 ZIP을 내려받으세요.')